# Fig S3B / S3C — Smaller target population (data processing)

A sketch approximating a smaller target population (n ~ several thousand) saturates sooner.

**Outputs:**
- `figure_data/FigS3B.LD_Sketch_MSE_Small_Target.rds` — LD error of sketch vs. random subset across B
- `figure_data/FigS3C.Sketch_Dim_FDR_Small_Target.rds` — fine-mapping FDR across B in the smaller design

In [1]:
source("../../R/utils.R")

## Fig S4B: LD error, smaller target population

In [2]:
res <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B_Over_Saturate/ld_mismatch/ld_mismatch_combined.rds")

## same metrics as Fig 2B / S2A (mse, max_abs_diff, concord, ...), against the target (n = 2,000) in-sample LD
metrics_long <- res %>%
  filter(!is.na(mse)) %>%
  mutate(
    resource = recode(reference, sketch = "LD Sketch", subset = "Random Subset"),
    resource = factor(resource, levels = names(pal_ld_resource))
  )
stopifnot(!any(is.na(metrics_long$resource)))

save_data(metrics_long, "FigS3B.LD_Sketch_MSE_Small_Target")

## Fig S4C: fine-mapping FDR, smaller GWAS design

In [3]:
res <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKBB_B_Over_Saturate_finemap/summary/result_susie_cs_summary_summary.rds")

## ---- label order (palette pal_sketch_dim in R/utils.R) ----
method_levels <- names(pal_sketch_dim)

## method token (fit names in 26 [target_finemap] = 05 [dim_finemap]) -> palette label
method_recode <- c(
  insample   = "SuSiE-RSS (In-sample LD)",
  sketch_mix = "SuSiE-rRSS (sketch)",
  subset_mix = "SuSiE-rRSS (subset)",
  sketch     = "SuSiE-RSS (sketch)",
  subset     = "SuSiE-RSS (subset)"
)

n_sim <- 2000   # sample_size in 26 [target_ld]; NCP = n_sim * h2g = 120 / 180 / 240

dat <- res %>%
  filter(method %in% names(method_recode)) %>%            # drops the *_SER rows
  mutate(
    method  = factor(recode(method, !!!method_recode), levels = method_levels),
    ncp     = round(n_sim * h2g),
    B       = as.integer(B_stoc),
    ncausal = as.integer(ncausal)
  )

In [4]:
## ---- FDR + bootstrap 95% CI, grouped by B as well ----
boot_cell_fdr <- function(d, key = NULL, B = 1000) {
  tot <- sum(d$total_set_number); tru <- sum(d$true_set_number)
  fdr_hat <- if (tot == 0) NA_real_ else 1 - tru / tot
  if (nrow(d) < 2 || tot == 0) return(tibble(fdr = fdr_hat, lo = NA_real_, hi = NA_real_))
  n  <- nrow(d)
  bs <- replicate(B, {
    idx <- sample.int(n, n, replace = TRUE)
    s   <- sum(d$total_set_number[idx])
    if (s == 0) NA_real_ else 1 - sum(d$true_set_number[idx]) / s
  })
  tibble(fdr = fdr_hat,
         lo  = quantile(bs, 0.025, na.rm = TRUE),
         hi  = quantile(bs, 0.975, na.rm = TRUE))
}

set.seed(1)
fdr_df <- dat %>%
  group_by(ncp, B, ncausal, method) %>%
  group_modify(~ boot_cell_fdr(.x)) %>%
  ungroup() %>% filter(B %in% c(500, 1000, 5000, 10000, 20000))

save_data(fdr_df, "FigS3C.Sketch_Dim_FDR_Small_Target")